In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
try:
    import lm_eval
except ImportError:
    %pip install -q git+https://github.com/EleutherAI/lm-evaluation-harness
    import lm_eval

# Constants

In [ ]:
TAWJEEH_DATASET_NAME = 'AraBench_dev'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/arabench_dev_experimental'
TASK_NAME='dialect_identification'
MODEL_PATH = "/hdd/shared_models/jais-13b-chat"
TUNED_MODEL_PATH = None
BATCH_SIZE=32

In [4]:
MODEL_NAME = MODEL_PATH.split('/')[-1]
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [ ]:
import requests
 
from tqdm.auto import tqdm
 
prompts = None
 
tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://tawjeeh.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts:
    raise Exception('Failed to fetch prompts')
prompts

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14796,
  'tags': [],
  'name': 'Text Classification-Act.As.Expert',
  'task': {'name': 'text classification'},
  'status': 'SUBMITTED',
  'template': "Task: Imagine you're organizing content into categories. Based on the details in the text, classify it into one of the following categories: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %}, {% endif %}{% endfor %}.\r\n\r\nText: {{ text }}\r\n\r\nPlease respond only with the category that best describes this text.\r\n|||\r\n{{ answer_choices[label] }}",
  'dataset_name': 'arbml/ArCovidVac',
  'dataset_subset': 'default',
  'answer_choices': ['celebrity',
   'info_news',
   'personal',
   'unrelated',
   'plan',
   'requests',
   'others',
   'rumors',
   'advice',
   'restrictions'],
  'text_direction': 'ltr'},
 {'id': 14795,
  'tags': [],
  'name': 'Topic Classification-Act.As.Expert',
  'task': {'name': 'topic classification'},
  'status': 'SUBMITTED',
  'template': 'Task: Imagine you’re categorizing article

filter prompts:
- get only the approved ones
- get only the ones on the sarcasim detection datasets (emotone_ar,sem_eval_2018_task_1)

In [6]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED', prompts))
len(filtered_prompts)

153

### Get the dataset prompts

In [7]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

2

### Download the dataset

In [8]:
import datasets

In [9]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['arabic', 'english', 'label'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['arabic', 'english', 'label'],
        num_rows: 10000
    })
})

### Merge the prompts

In [10]:
from jinja2 import Environment, StrictUndefined

In [11]:
def apply_template(prompt_template, sample):
    template = prompt_template['template']
    sample['answer_choices'] = prompt_template['answer_choices']
    env = Environment(undefined=StrictUndefined)
    if "|||" not in template:
        raise ValueError("No ||| dividor")
    template = env.from_string(template)
    rendered_template = template.render(**sample)
    return rendered_template

Perform generation on one example prompt, for experimentation

In [12]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][2]))

For the following Arabic text: هيدا صالح ل اربعتاعشر يوم، فا فيك تستعملو لحد يوم تلاتة و عشرين., the most probable dialect based on the spelling variations that represent dialectal pronunciation, among the following dialects: Tunisian, MSA, Moroccan, Qatari, Egyptian, Lebanese is: 
|||
Lebanese


merge prompts

In [13]:
for prompt in dataset_prompts:
    prompt['merged_samples'] = list(
        map(
            lambda sample: apply_template(prompt, sample),
            hf_exp_dataset['test'],
        )
    )

# Evaluate on each prompt and report the results

In [14]:
from datasets import DatasetDict

def create_hf_dataset(examples, columns = ['text', 'label']):
  texts = []
  labels = []
  for example in examples:
    prefix= example.split('|||')[0].replace('\n', '')
    output = example.split('|||')[1].replace('\n', '')
    texts.append(prefix)
    labels.append(output)
  dataset = DatasetDict({ 'test' : datasets.Dataset.from_dict({
      columns[0]: texts,
      columns[1]: labels,
  })})
  return dataset

In [ ]:
from lm_eval.models.huggingface import HFLM
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, trust_remote_code=True, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

In [ ]:
from peft import PeftModel
if TUNED_MODEL_PATH:
    print('loading tuned model')
    model = PeftModel.from_pretrained(model,TUNED_MODEL_PATH)

In [ ]:
lm_obj = HFLM(
    pretrained=model,
    trust_remote_code=True,
    # parallelize=True,
    device_map="auto",
    tokenizer=tokenizer,
    batch_size=BATCH_SIZE,
)

2024-11-06:18:25:19,979 INFO     [huggingface.py:494] Model type cannot be determined. Using default model type 'default'
2024-11-06:18:25:20,269 INFO     [huggingface.py:350] Model parallel was set to True, setting max memory per GPU to {0: 84523417600} and device map to 'auto'


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [17]:
def evaluate_tasks(tasks,dataset_sub_path=TAWJEEH_DATASET_NAME):
    # MAKE SURE THE NOTEBOOK IS RUNNING FROM THE PROJECT ROOT!
    task_manager = lm_eval.tasks.TaskManager(include_path=f"eval_harness_extra_tasks/{dataset_sub_path}")
    results = lm_eval.simple_evaluate(  # call simple_evaluate
        model=lm_obj,
        tasks=tasks,
        num_fewshot=0,
        task_manager=task_manager,
    )
    return results

In [ ]:
import json

def create_and_evaluate_single_prompt(prompt):
    prompt_id = prompt['id']
    if TUNED_MODEL_PATH:
        results_dir = f'evaluation_results/{MODEL_NAME}_tuned/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    else:
      results_dir = f'evaluation_results/{MODEL_NAME}/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    prompt_results_file_path = f'{results_dir}/prompt_{prompt_id}.json'
    
    # Skip if results already exist
    if os.path.exists(prompt_results_file_path) and os.path.getsize(prompt_results_file_path) > 0:
        print(f"Skipping prompt {prompt_id} - results already exist")
        with open(prompt_results_file_path, 'r') as f:
            return json.load(f)
    
    # Create dataset and task files
    merged_samples = prompt['merged_samples']
    answer_choices = prompt['answer_choices']
    dataset = create_hf_dataset(merged_samples)
    
    # Save dataset
    dataset_dir = f'experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}'
    os.makedirs(dataset_dir, exist_ok=True)
    dataset['test'].to_parquet(f"{dataset_dir}/data.parquet")
    
    # Create YAML configuration
    yaml_text = f'''task: {TAWJEEH_DATASET_NAME}_prompt_{prompt_id}
dataset_path: experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}
output_type: multiple_choice
test_split: train
doc_to_text: text
doc_to_target: label
doc_to_choice: {answer_choices}
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: True
  - metric: acc_norm
    aggregation: mean
    higher_is_better: true
metadata:
  version: 1.0'''
    
    # Save YAML
    yaml_dir = f'eval_harness_extra_tasks/{TAWJEEH_DATASET_NAME}'
    os.makedirs(yaml_dir, exist_ok=True)
    with open(f'{yaml_dir}/prompt_{prompt_id}.yaml', 'w') as f:
        f.write(yaml_text)
    
    # Evaluate single prompt
    evaluation_task_name = f'{TAWJEEH_DATASET_NAME}_prompt_{prompt_id}'
    prompt_results = evaluate_tasks(tasks=[evaluation_task_name])
    
    print(lm_eval.utils.make_table(prompt_results))
    
    # Save results immediately
    os.makedirs(results_dir, exist_ok=True)
    with open(prompt_results_file_path, 'w') as f:
        json.dump(prompt_results, f, ensure_ascii=False, indent=4, 
                 default=lambda o: '<not serializable>')
    
    print(f"Completed evaluation for prompt {prompt_id}")
    return prompt_results

In [19]:
def evaluate_all_prompts_sequentially(dataset_prompts):
    print(f"Starting sequential evaluation of {len(dataset_prompts)} prompts")
    all_results = {}
    
    for i, prompt in enumerate(dataset_prompts, 1):
        print('-' * 80)
        print(f"\nProcessing prompt {i}/{len(dataset_prompts)}")
        print("Template:", prompt['template'])
        print('-' * 80)
        
        prompt_results = create_and_evaluate_single_prompt(prompt)
        all_results[f"{TAWJEEH_DATASET_NAME}_prompt_{prompt['id']}"] = prompt_results
    
    return {'results': all_results}

In [20]:
evaluate_all_prompts_sequentially(dataset_prompts=dataset_prompts)

Starting sequential evaluation of 2 prompts
--------------------------------------------------------------------------------

Processing prompt 1/2
Template: For the following Arabic text: {{arabic}}, the most probable dialect based on the spelling variations that represent dialectal pronunciation, among the following dialects: Tunisian, MSA, Moroccan, Qatari, Egyptian, Lebanese is: 
|||
{{answer_choices[label]}}
--------------------------------------------------------------------------------


Creating parquet from Arrow format:   0%|          | 0/10 [00:00<?, ?ba/s]

2024-11-06:18:26:45,466 INFO     [evaluator.py:164] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2024-11-06:18:26:45,471 INFO     [evaluator.py:217] Using pre-initialized model


Generating train split: 0 examples [00:00, ? examples/s]

2024-11-06:18:26:45,545 WARNING  [task.py:325] [Task: AraBench_dev_prompt_14782] has_training_docs and has_validation_docs are False, using test_docs as fewshot_docs but this is not recommended.
2024-11-06:18:26:45,546 WARNING  [task.py:325] [Task: AraBench_dev_prompt_14782] has_training_docs and has_validation_docs are False, using test_docs as fewshot_docs but this is not recommended.
2024-11-06:18:26:45,718 WARNING  [evaluator.py:270] Overwriting default num_fewshot of AraBench_dev_prompt_14782 from None to 0
2024-11-06:18:26:45,719 INFO     [task.py:415] Building contexts for AraBench_dev_prompt_14782 on rank 0...
100%|██████████| 10000/10000 [00:00<00:00, 28131.59it/s]
2024-11-06:18:26:46,264 INFO     [evaluator.py:489] Running loglikelihood requests
Running loglikelihood requests: 100%|██████████| 60000/60000 [1:12:26<00:00, 13.81it/s]
2024-11-06:19:39:31,030 WARNING  [huggingface.py:1375] Failed to get model SHA for /hdd/shared_models/jais-13b-chat at revision main. Error: Repo 

...

2024-11-06:19:39:46,174 INFO     [evaluator.py:164] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2024-11-06:19:39:46,175 INFO     [evaluator.py:217] Using pre-initialized model


Generating train split: 0 examples [00:00, ? examples/s]

2024-11-06:19:39:46,253 WARNING  [task.py:325] [Task: AraBench_dev_prompt_14561] has_training_docs and has_validation_docs are False, using test_docs as fewshot_docs but this is not recommended.
2024-11-06:19:39:46,254 WARNING  [task.py:325] [Task: AraBench_dev_prompt_14561] has_training_docs and has_validation_docs are False, using test_docs as fewshot_docs but this is not recommended.
2024-11-06:19:39:46,484 WARNING  [evaluator.py:270] Overwriting default num_fewshot of AraBench_dev_prompt_14561 from None to 0
2024-11-06:19:39:46,485 INFO     [task.py:415] Building contexts for AraBench_dev_prompt_14561 on rank 0...
100%|██████████| 10000/10000 [00:00<00:00, 22279.22it/s]
2024-11-06:19:39:47,180 INFO     [evaluator.py:489] Running loglikelihood requests
Running loglikelihood requests: 100%|██████████| 60000/60000 [56:16<00:00, 17.77it/s] 
2024-11-06:20:36:20,325 WARNING  [huggingface.py:1375] Failed to get model SHA for /hdd/shared_models/jais-13b-chat at revision main. Error: Repo i

|          Tasks          |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|-------------------------|------:|------|-----:|--------|---|-----:|---|-----:|
|AraBench_dev_prompt_14561|      1|none  |     0|acc     |↑  |0.1344|±  |0.0034|
|                         |       |none  |     0|acc_norm|↑  |0.1333|±  |0.0034|

Completed evaluation for prompt 14561


{'results': {'AraBench_dev_prompt_14782': {'results': {'AraBench_dev_prompt_14782': {'acc,none': 0.2163,
     'acc_stderr,none': 0.004117417432399934,
     'acc_norm,none': 0.1577,
     'acc_norm_stderr,none': 0.0036447770082603083}},
   'group_subtasks': {'AraBench_dev_prompt_14782': []},
   'configs': {'AraBench_dev_prompt_14782': {'task': 'AraBench_dev_prompt_14782',
     'dataset_path': 'experimental_hf_datasets/AraBench_dev/prompt_14782',
     'test_split': 'train',
     'doc_to_text': 'text',
     'doc_to_target': 'label',
     'doc_to_choice': ['Tunisian',
      'MSA',
      'Morrocan',
      'Qatari',
      'Egyptian',
      'Lebanese'],
     'description': '',
     'target_delimiter': ' ',
     'fewshot_delimiter': '\n\n',
     'num_fewshot': 0,
     'metric_list': [{'metric': 'acc',
       'aggregation': 'mean',
       'higher_is_better': True},
      {'metric': 'acc_norm', 'aggregation': 'mean', 'higher_is_better': True}],
     'output_type': 'multiple_choice',
     'repeats